# MAPF Solver Log Analysis

This notebook consolidates the LNS and WholeSolve experiment logs and produces publication-grade figures that highlight solver performance across maps, categories, and overall runtime distributions.


## Environment and helper utilities

The cell below imports the analysis dependencies, configures Matplotlib for a publication-ready aesthetic, and defines helper functions that are reused throughout the notebook.


In [3]:
from __future__ import annotations

import math
from pathlib import Path
from typing import Iterable, Optional

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib import cm
from matplotlib import colors as mcolors
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

plt.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 11,
    "figure.dpi": 120,
})

for style in ("seaborn-v0_8-whitegrid", "seaborn-whitegrid", "ggplot"):
    try:
        plt.style.use(style)
        break
    except OSError:
        continue

SOLVER_COLORS: dict[str, str] = {
    "LNS": "#2F4B7C",
    "WholeSolve": "#F95D6A",
}
SOLVER_STYLES: dict[str, str] = {
    "LNS": "-",
    "WholeSolve": "--",
}
CATEGORY_PALETTE: dict[str, str] = {
    "Urban": "#4E79A7",
    "Rooms": "#F28E2B",
    "Warehouse": "#59A14F",
    "Empty": "#E15759",
    "Uncategorised": "#9D7660",
}

def lighten_color(color: str, amount: float = 0.45) -> str:
    """Blend ``color`` with white to obtain a lighter variant."""
    amount = max(0.0, min(1.0, amount))
    r, g, b = mcolors.to_rgb(color)
    return mcolors.to_hex((
        1 - (1 - r) * (1 - amount),
        1 - (1 - g) * (1 - amount),
        1 - (1 - b) * (1 - amount),
    ))

LIGHT_SOLVER_COLORS = {
    name: lighten_color(color, 0.55) for name, color in SOLVER_COLORS.items()
}

def locate_log_root(candidates: Optional[Iterable[Path]] = None) -> Path:
    """Return the first existing log directory from ``candidates``."""
    search_paths = list(candidates) if candidates is not None else [
        Path("lns_clean/logs"),
        Path("../lns_clean/logs"),
        Path("../../lns_clean/logs"),
    ]
    for path in search_paths:
        if path.exists():
            return path.resolve()
    raise FileNotFoundError(
        "Could not locate the 'lns_clean/logs' directory relative to this notebook."
    )

def infer_map_category(map_name: Optional[str], map_path: Optional[str]) -> str:
    """Infer a semantic map category from directory and path hints."""
    name = (map_name or "").lower()
    path = (map_path or "").lower()
    if "warehouse" in name or "warehouse" in path:
        return "Warehouse"
    if "room" in name or "room" in path:
        return "Rooms"
    if "empty" in name or "empty" in path:
        return "Empty"
    if name in {"berlin", "paris"}:
        return "Urban"
    return "Uncategorised"

def humanize_map_id(map_id: Optional[str]) -> str:
    """Render a compact yet human-readable label for a map identifier."""
    if map_id is None or (isinstance(map_id, float) and math.isnan(map_id)):
        return "Unknown map"
    tokens = str(map_id).replace("_", "-").split("-")
    tokens = [token for token in tokens if token]
    if not tokens:
        return "Unknown map"
    base_name = tokens[0]
    name_lookup = {
        "room": "Rooms",
        "rooms": "Rooms",
        "empty": "Empty",
        "warehouse": "Warehouse",
        "berlin": "Berlin",
        "paris": "Paris",
    }
    label = name_lookup.get(base_name.lower(), base_name.replace("_", " ").title())
    dims = [token for token in tokens[1:] if token]
    if dims:
        return f"{label} {'×'.join(dims)}"
    return label

def build_cdf(frame: pd.DataFrame, group_cols: list[str], value_col: str = "runtime_s") -> pd.DataFrame:
    """Compute an empirical CDF for ``value_col`` grouped by ``group_cols``."""
    results = []
    for keys, group in frame.groupby(group_cols):
        values = group[value_col].dropna().sort_values().reset_index(drop=True)
        if values.empty:
            continue
        cdf_values = (values.index + 1) / len(values)
        result = pd.DataFrame({value_col: values, "cdf": cdf_values})
        if not isinstance(keys, tuple):
            keys = (keys,)
        for column, key in zip(group_cols, keys):
            result[column] = key
        results.append(result)
    if results:
        ordered_cols = list(group_cols) + [value_col, "cdf"]
        return pd.concat(results, ignore_index=True)[ordered_cols]
    return pd.DataFrame(columns=list(group_cols) + [value_col, "cdf"])


In [6]:
log_root = locate_log_root()
from Data_analysis import load_solver_logs  # if available
raw_logs = load_solver_logs(base_dir=log_root)      # returns a DataFrame

In [9]:
log_root = Path("/home/max/MAPF_LNS_SAT/lns_clean/logs")
raw = load_solver_logs(log_root)
exp = raw[raw["source_file"].astype(str).str.endswith("experiments.csv")].copy()
exp["runtime_s"] = pd.to_numeric(exp["total_runtime_ms"], errors="coerce") / 1000
exp["is_solved"] = pd.to_numeric(exp["solved"], errors="coerce").eq(1)
cdf = build_cdf(exp[exp["is_solved"]], ["solver"])
cdf.head()

,solver,runtime_s,cdf
0,LNS,0.003485,0.000381
1,LNS,0.005791,0.000762
2,LNS,0.006357,0.001143
3,LNS,0.006530,0.001524
4,LNS,0.006615,0.001905
